<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/PySpark-RDD-Analytics---AirBnb-NYC-2019-Dataset/blob/main/AirBnb_rdd_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("AirBnb_NYC_RDD_Analysis") \
    .getOrCreate()

print(spark)

In [2]:
sc = spark.sparkContext
print(sc)

<SparkContext master=local appName=AirBnb_NYC_RDD_Analysis>


# 🏠 PySpark RDD Analytics — AirBnb NYC 2019 Dataset

**Domain:** Short-Term Rental Analytics | **Engine:** Apache Spark (RDD API) | **Language:** Python 3

---

## 📋 Dataset Schema

| Field | Index | Type | Description |
|---|---|---|---|
| `id` | 0 | int | Unique listing identifier |
| `name` | 1 | str | Listing name |
| `host_id` | 2 | int | Unique host identifier |
| `host_name` | 3 | str | Host name |
| `neighbourhood_group` | 4 | str | Borough (Manhattan, Brooklyn, etc.) |
| `neighbourhood` | 5 | str | Neighbourhood name |
| `latitude` | 6 | float | Latitude coordinate |
| `longitude` | 7 | float | Longitude coordinate |
| `room_type` | 8 | str | Room type: Private room / Entire home/apt / Shared room |
| `price` | 9 | int | Price per night in USD |
| `minimum_nights` | 10 | int | Minimum nights required |
| `number_of_reviews` | 11 | int | Total number of reviews |
| `last_review` | 12 | str | Date of last review |
| `reviews_per_month` | 13 | float | Average reviews per month |
| `calculated_host_listings_count` | 14 | int | Total listings by this host |
| `availability_365` | 15 | int | Days available per year |

**Sample record:**
`2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,19-10-2018,0.21,6,365`

In [3]:
import csv

# Load raw CSV, strip header, parse each row with csv.reader to handle quoted fields
raw_rdd = sc.textFile("/content/sample_data/AB_NYC_2019.csv")
header  = raw_rdd.first()

data_rdd = raw_rdd \
    .filter(lambda row: row != header) \
    .map(lambda row: next(csv.reader([row])))

print(f"Total listings: {data_rdd.count()}")
data_rdd.take(2)

Total listings: 49080


[['2539',
  'Clean & quiet apt home by the park',
  '2787',
  'John',
  'Brooklyn',
  'Kensington',
  '40.64749',
  '-73.97237',
  'Private room',
  '149',
  '1',
  '9',
  '19-10-2018',
  '0.21',
  '6',
  '365'],
 ['2595',
  'Skylit Midtown Castle',
  '2845',
  'Jennifer',
  'Manhattan',
  'Midtown',
  '40.75362',
  '-73.98377',
  'Entire home/apt',
  '225',
  '1',
  '45',
  '21-05-2019',
  '0.38',
  '2',
  '355']]

In [4]:
# Helper: safely parse float fields (reviews_per_month may be empty)
def safe_float(value):
    try:
        return float(value)
    except (ValueError, TypeError):
        return None

In [5]:
# Q1 — Which host has the most number of listed rooms?
# col[2] = host_id, col[3] = host_name
host_with_most_listings_rdd = data_rdd \
    .filter(lambda col: len(col) > 3) \
    .map(lambda col: ((col[2], col[3]), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

top_host_by_listings = host_with_most_listings_rdd.first()
print(f"Host with most listings → ID: {top_host_by_listings[0][0]}, "
      f"Name: {top_host_by_listings[0][1]}, "
      f"Listings: {top_host_by_listings[1]}")

Host with most listings → ID: 219517861, Name: Sonder (NYC), Listings: 327


In [6]:
# Q2 — Which host has the highest reviews_per_month (used as rating proxy)?
# col[13] = reviews_per_month
highest_rated_host = data_rdd \
    .filter(lambda col: len(col) > 13) \
    .filter(lambda col: col[13] not in ('', '0')) \
    .map(lambda col: ((col[2], col[3]), safe_float(col[13]))) \
    .filter(lambda x: x[1] is not None) \
    .max(key=lambda x: x[1])

print(f"Highest rated host → ID: {highest_rated_host[0][0]}, "
      f"Name: {highest_rated_host[0][1]}, "
      f"Reviews/month: {highest_rated_host[1]}")

Highest rated host → ID: 244361589, Name: Row NYC, Reviews/month: 58.5


In [7]:
# Q3 — Which host has the minimum reviews_per_month?
# col[13] = reviews_per_month
lowest_rated_host = data_rdd \
    .filter(lambda col: len(col) > 13) \
    .filter(lambda col: col[13] not in ('', '0')) \
    .map(lambda col: ((col[2], col[3]), safe_float(col[13]))) \
    .filter(lambda x: x[1] is not None) \
    .min(key=lambda x: x[1])

print(f"Lowest rated host → ID: {lowest_rated_host[0][0]}, "
      f"Name: {lowest_rated_host[0][1]}, "
      f"Reviews/month: {lowest_rated_host[1]}")

Lowest rated host → ID: 140025, Name: Fredah, Reviews/month: 0.01


In [8]:
# Q4 — Print the first 5 highest rated hosts (by reviews_per_month)
# Aggregate max reviews_per_month per host, then take top 5
top_5_highest_rated_hosts = data_rdd \
    .filter(lambda col: len(col) > 13) \
    .filter(lambda col: col[13] not in ('', '0')) \
    .map(lambda col: ((col[2], col[3]), safe_float(col[13]))) \
    .filter(lambda x: x[1] is not None) \
    .reduceByKey(lambda a, b: max(a, b)) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(5)

print("Top 5 Highest Rated Hosts:")
for rank, (host, rating) in enumerate(top_5_highest_rated_hosts, 1):
    print(f"  {rank}. ID: {host[0]}, Name: {host[1]}, Reviews/month: {rating}")

Top 5 Highest Rated Hosts:
  1. ID: 244361589, Name: Row NYC, Reviews/month: 58.5
  2. ID: Anting, Name: Manhattan, Reviews/month: 33.0
  3. ID: Anting, Name: Brooklyn, Reviews/month: 33.0
  4. ID: 228415932, Name: Louann, Reviews/month: 20.94
  5. ID: 156684502, Name: Nalicia, Reviews/month: 19.75


In [9]:
# Q5 — Print the first 5 lowest rated hosts (by reviews_per_month)
# Aggregate min reviews_per_month per host, then take bottom 5
top_5_lowest_rated_hosts = data_rdd \
    .filter(lambda col: len(col) > 13) \
    .filter(lambda col: col[13] not in ('', '0')) \
    .map(lambda col: ((col[2], col[3]), safe_float(col[13]))) \
    .filter(lambda x: x[1] is not None) \
    .reduceByKey(lambda a, b: min(a, b)) \
    .sortBy(lambda x: x[1], ascending=True) \
    .take(5)

print("Top 5 Lowest Rated Hosts:")
for rank, (host, rating) in enumerate(top_5_lowest_rated_hosts, 1):
    print(f"  {rank}. ID: {host[0]}, Name: {host[1]}, Reviews/month: {rating}")

Top 5 Lowest Rated Hosts:
  1. ID: 31374, Name: Shon, Reviews/month: 0.01
  2. ID: 140025, Name: Fredah, Reviews/month: 0.01
  3. ID: 204539, Name: Mark, Reviews/month: 0.01
  4. ID: 394752, Name: Allison, Reviews/month: 0.01
  5. ID: 507304, Name: Derrick, Reviews/month: 0.01


In [10]:
# Q6 — How many rooms are available for all 365 days?
# col[15] = availability_365
rooms_available_365_days_count = data_rdd \
    .filter(lambda col: len(col) > 15) \
    .filter(lambda col: col[15].strip() == '365') \
    .count()

print(f"Rooms available for 365 days: {rooms_available_365_days_count}")

Rooms available for 365 days: 1285


In [11]:
# Q7 — How many Private rooms are available?
# col[8] = room_type
private_rooms_count = data_rdd \
    .filter(lambda col: len(col) > 8) \
    .filter(lambda col: col[8].strip() == 'Private room') \
    .count()

print(f"Total Private rooms available: {private_rooms_count}")

Total Private rooms available: 22229


In [12]:
# Q8 — How many Entire home/apt listings are available?
# col[8] = room_type
entire_home_apt_count = data_rdd \
    .filter(lambda col: len(col) > 8) \
    .filter(lambda col: col[8].strip() == 'Entire home/apt') \
    .count()

print(f"Total Entire home/apt available: {entire_home_apt_count}")

Total Entire home/apt available: 25348
